# Properties of Time Series

*Based on: Hyndman, R.J. & Athanasopoulos, G., Forecasting: Principles and Practice (3rd ed.), Chapter 1*

---


## Learning Objectives

At the end of this notebook, you should be able to:

- Recognise time-series data and distinguish it from cross-sectional data.
- Identify the **frequency** of a series and name common frequencies (annual, quarterly, monthly, daily).
- Describe the **forecasting task** and the meaning of the forecast horizon $h$.
- Represent a time series in Python using a `pandas.Series` with a `DatetimeIndex`.
- Identify the visual signatures of **trend**, **seasonality**, **cycles**, and **noise** in real-world series.


## 1.1 Definition

A **time series** is a sequence of observations recorded at successive points in time,
typically at equally spaced intervals.

Formally, we denote a time series as:

$$y_1, y_2, \ldots, y_T$$

where $y_t$ is the observation at time $t$, and $T$ is the total number of observations.

Key characteristics:
- Observations are **ordered**: the sequence matters.
- Successive observations are typically **not independent** (yesterday's value influences today's).
- The goal is often to **forecast** future values $y_{T+1}, y_{T+2}, \ldots$

> The simplest time series models assume that observations are independent...
> but the whole point of time series analysis is that this assumption is usually violated.
> Hyndman & Athanasopoulos, FPP3


## 1.2 Time Series vs Cross-Sectional Data

| | **Cross-sectional data** | **Time series data** |
|---|---|---|
| **What varies** | Units / individuals | Time |
| **Example** | Heights of 100 people at one point in time | One person's height measured every year |
| **Key assumption** | Observations are often i.i.d. | Observations are ordered and correlated |
| **Typical methods** | Regression, clustering | ARIMA, exponential smoothing, etc. |

In time series, **when** something was measured is just as important as **what** was measured.


## 1.3 Frequency and Regularity

Time series can be recorded at many frequencies:

| Frequency | Periods per year | Example |
|---|---|---|
| Annual | 1 | GDP per country |
| Quarterly | 4 | Corporate earnings |
| Monthly | 12 | Retail sales |
| Weekly | 52 | Flu cases |
| Daily | 365 | Stock prices |
| Hourly | 8,760 | Electricity demand |

Observations can also be **irregular** (e.g., timestamps of website clicks), which requires special handling.


## 1.4 Examples Across Domains

Time series arise everywhere:

- **Economics & Finance**: GDP, inflation, stock prices, exchange rates
- **Business**: sales, inventory, web traffic, customer counts
- **Environment**: temperature, rainfall, CO2 concentration
- **Medicine & Health**: heart rate, disease incidence, hospital admissions
- **Engineering**: sensor readings, network traffic, vibration signals

Below we load and visualise four real-world examples.


In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (11, 3.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Libraries loaded successfully.')


Load four real-world series spanning four domains. Each is a `pandas.Series` (or DataFrame) indexed by time.

In [ ]:
# Load four classic time series from statsmodels
from statsmodels.datasets import co2, sunspots, elec_equip, get_rdataset

# 1. Atmospheric CO2 (weekly, 1958-2001) - resampled to monthly
co2_data = co2.load_pandas().data.resample('MS').mean()

# 2. Annual sunspot numbers (1700-2008)
sun_data = sunspots.load_pandas().data.set_index('YEAR')
sun_data.index = pd.to_datetime(sun_data.index, format='%Y')

# 3. International airline passengers (monthly, 1949-1960)
air = get_rdataset('AirPassengers').data
air.index = pd.date_range('1949-01', periods=len(air), freq='MS')
air = air['value']

# 4. Electrical equipment orders in Euro area (monthly, 1996-2012)
equip = elec_equip.load_pandas().data

print('Datasets ready.')


Plot all four side by side. Use the observation prompts below the figure to spot trend, seasonality, cycles, and noise.

In [ ]:
# Plot: four real-world time series
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('Four Real-World Time Series', fontsize=14, fontweight='bold', y=1.01)

axes[0, 0].plot(co2_data, color='steelblue', linewidth=0.9)
axes[0, 0].set_title('Atmospheric CO2 (Mauna Loa, monthly)')
axes[0, 0].set_ylabel('ppm')

axes[0, 1].plot(sun_data, color='orange', linewidth=0.8)
axes[0, 1].set_title('Annual Sunspot Numbers')
axes[0, 1].set_ylabel('Count')

axes[1, 0].plot(air, color='seagreen', linewidth=1.1)
axes[1, 0].set_title('International Airline Passengers (monthly)')
axes[1, 0].set_ylabel('Thousands')

axes[1, 1].plot(equip, color='crimson', linewidth=0.9)
axes[1, 1].set_title('Euro-area Electrical Equipment Orders (monthly)')
axes[1, 1].set_ylabel('Index (2005 = 100)')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


**Observation prompts**: before moving on, note:
1. Which series has a clear *upward trend*?
2. Which series shows *periodic up-and-down cycles*?
3. Which series shows both a trend *and* regular seasonal peaks?
4. Which series appears most *irregular* (volatile)?

We will formalise these patterns (trend, seasonality, cycles, noise) in the next section.


## 1.5 The Forecasting Task

Given observations up to time $T$, we want to produce a **forecast** $\hat{y}_{T+h|T}$,
our best estimate of $y_{T+h}$ given everything observed up to $T$.

- $h$ is called the **forecast horizon** (h=1: one-step-ahead, h=12: twelve steps ahead, etc.)
- Forecasting is **not the same** as interpolation, we are estimating *unseen future* values.
- Good forecasting requires understanding what patterns exist and how stable they are.

The difficulty of forecasting depends on:
- How well we understand the underlying system
- How much data is available
- Whether the future resembles the past (stationarity)
- The forecast horizon $h$


In [ ]:
# Illustration: observed vs forecast region
fig, ax = plt.subplots(figsize=(11, 4))

train = air[:'1957']
future_idx = pd.date_range('1958-01', periods=36, freq='MS')

ax.plot(train, color='steelblue', label='Observed (training)')
ax.axvspan(future_idx[0], future_idx[-1], alpha=0.12, color='orange')
ax.annotate('Observed data', xy=(pd.Timestamp('1951-01'), 420),
            fontsize=10, color='steelblue')
ax.annotate('Forecast horizon (h = 1...36)', xy=(pd.Timestamp('1958-02'), 420),
            fontsize=10, color='darkorange')
ax.set_title('The Forecasting Task: Using the Past to Predict the Future')
ax.set_ylabel('Passengers (thousands)')
ax.legend()
plt.tight_layout()
plt.show()


## 1.6 Time Series Objects in Python

In Python the natural container for a time series is a **`pandas.Series` with a `DatetimeIndex`**.


In [ ]:
# Building a time series by hand
dates = pd.date_range(start='2020-01-01', periods=6, freq='MS')
values = [112, 118, 132, 129, 121, 135]

ts = pd.Series(values, index=dates, name='sales')
print(ts)
print(f'\nData type of index : {type(ts.index)}')
print(f'Frequency inferred : {ts.index.freq}')


A few one-liners you will reach for again and again, slicing by year, computing differences, rolling means.

In [ ]:
# Useful time series operations

# Slicing by date string
print('--- Data from March 2020 onwards ---')
print(ts['2020-03':])

# Lag (shift)
print('\n--- Lag-1 (previous month) ---')
print(ts.shift(1))

# First difference
print('\n--- First difference (change month-on-month) ---')
print(ts.diff())


Resampling moves a series to a coarser frequency (downsampling, pick an aggregation) or a finer one (upsampling, pick a fill rule).

In [ ]:
# Resampling: changing frequency
annual = air.resample('YE').sum()   # monthly -> annual totals
print('Annual totals (airline passengers):')
print(annual)


---
## Exercises

**Exercise 1.1**: Load any time series of your choice and plot it.
Write two sentences describing what you see.

Suggested datasets:
```python
from statsmodels.datasets import get_rdataset
df = get_rdataset('lynx').data          # Canadian lynx trappings
df = get_rdataset('UKDriverDeaths').data  # UK road deaths
```


In [ ]:
# Your code here



*Your observations:*

1. 
2. 


**Exercise 1.2**: For the airline passengers series (`air`), compute:
- The month-on-month *first difference*
- The year-on-year *percentage change* (hint: `air.pct_change(12) * 100`)

Plot both derived series. What do they reveal that the original series does not?


In [ ]:
# Your code here



**Exercise 1.3**: Answer the following conceptual questions:

a. Is a sequence of daily maximum temperatures in a city over one year a time series?
   What about temperatures measured at random times?

b. A company measures the sales of 500 products in the same month.
   Is this a time series?

c. You have daily stock prices for Apple from 2010 to 2024.
   If you want to forecast 30 days ahead, what is T and what is h?


*Your answers:*

a. 

b. 

c. 


---
## Key Takeaways

| Concept | Summary |
|---|---|
| Time series | Ordered sequence of observations indexed by time |
| vs Cross-sectional | Order matters; observations are correlated across time |
| Frequency | Annual / quarterly / monthly / daily / ... |
| Forecasting | Estimating $y_{T+h}$ using $y_1, \ldots, y_T$ |
| Python tool | `pd.Series` with `DatetimeIndex` |

---
**Next section: Time Series Patterns: Trend, Seasonality, Cycles, and Noise**
